# Final Project: Train an LLM as a Jev-Like Decision Model

The final project adapts a small pretrained LLM into a Jev-like structured decision model. Implement the data adapter, decision head, supervised training loop, calibration, and inference interface with PyTorch and its ecosystem. The same trained head must accept changing candidate labels and support Choice, Noul, and Score questions across tasks.

Unsloth's open decision-model implementation is a concrete architecture reference. The educational model makes its own documented simplifications and uses ordinary PyTorch operations. Jev's proprietary architecture and training process remain outside the available evidence. The final report describes an independently trained Jev-like model and its measured behavior.

## Project objective

Build a local, versioned Python package or service that accepts a state and one or more typed decision questions, then returns option probabilities or rubric scores. Train the head and adapt the pretrained LLM backbone through LoRA or full fine-tuning. Measure whether the resulting system transfers to a task excluded from adaptation.

Use PyTorch for model components and the training loop, Hugging Face Transformers for pretrained weights and tokenization, and PEFT when using LoRA. A small pretrained decoder such as [Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B) is a reasonable starting checkpoint. Pin its revision and check its license and model card. Build a tiny randomly initialized decoder for a CPU smoke run that checks tensor shapes and gradients; use the pretrained checkpoint for the actual task-transfer experiment.

“From scratch” means implementing the decision-model logic and understanding its objective around a pretrained backbone. It does not require reimplementing the backbone, autograd, an optimizer, or GPU kernels. The model must score the supplied options directly; free-form generated labels are a separate comparison.

## Start from the runnable implementation

Chapter 06 contains the [worked PyTorch implementation](06-finetuning.html): model code, attention and readout equations, the decision loss and its gradient, head warm-up, backbone/LoRA adaptation, and save/reload checks. The maintained package is `projects/text-decisions`. Use its working trainer as the starting point, then extend the data adapters, calibration and evaluation for this project.

From the repository root, first run the offline pipeline:

~~~bash
uv sync --all-packages
uv run --package text-decisions text-decisions train \
  --checkpoint tiny-qwen --head joint --adaptation full \
  --train projects/text-decisions/examples/train.jsonl \
  --validation projects/text-decisions/examples/validation.jsonl \
  --epochs 2 --output .tmp/decision-smoke
uv run --package text-decisions text-decisions predict \
  --model .tmp/decision-smoke \
  --request projects/text-decisions/examples/request.json
~~~

The hand-authored fixture covers sentiment and topic, all three question types, and disjoint source groups. It checks execution, target mapping and saved inference. Replace it with audited task datasets for quality or transfer claims.

For pretrained adaptation, use `--checkpoint Qwen/Qwen2.5-0.5B --adaptation lora`, pin `--revision`, and choose `--device cuda` when a suitable GPU is available. CPU execution is also supported. The trainer first warms up the head with frozen backbone weights, rebuilds the optimizer for adaptation, shuffles candidates with semantic targets preserved, and selects the saved checkpoint on validation loss. Switch `--head candidate` for the simpler scorer, or use `--checkpoint answerdotai/ModernBERT-base --adaptation full` for the encoder comparison.

Keep calibration and held-out-task evaluation separate from this starter trainer. A second optional baseline is AnyJev's training-free answer-token readout, explained with code and equations in Chapter 06. The custom-head model, a supervised label-token extension, and a training-free readout use different output parameters and training assumptions; label comparisons accordingly.

## Data contract

Choose at least three task families with different label spaces. A starter design trains on AG News topic classification and SST5 sentiment, with Banking77 intent classification reserved entirely for task-transfer evaluation. Inspect dataset licenses, provenance, official splits, label definitions, and class balance. Use IMDb as an additional sentiment comparison when useful.

Include labeled training questions for all three output types. For SST5, the same audited sentiment label can support a five-option Choice question, an ordered Score rubric, and a Noul question asking whether the sentiment is positive. Define the mapping explicitly: for labels ordered from very negative through very positive, only the two positive levels make that proposition true. Keep every derived question from one source example in the same split. Report how repeated questions affect record weighting.

Convert each example to a shared request-and-target shape. The `task_id` is split and reporting metadata. The model sees `state` and `questions`; `answers` is the training target.

~~~text
{
  "task_id": "sentiment",
  "state": "The movie was excellent.",
  "questions": {
    "sentiment": {
      "type": "choice",
      "instructions": "Determine the overall sentiment of the review.",
      "criteria": {
        "negative": "An overall unfavorable opinion.",
        "positive": "An overall favorable opinion."
      }
    }
  },
  "answers": {
    "sentiment": {"label": "positive"}
  }
}
~~~

Keep each trained task's training, validation, calibration, and test roles explicit. Split by source group where related examples or derived questions can cross rows. Use validation data to select checkpoints and use a disjoint calibration partition to fit temperatures. Reserve the held-out task from model fitting, checkpoint selection, prompt tuning, and temperature fitting. Test it only after those choices are fixed.

For an additional supervised-on-target comparison, train a task-specific baseline on the held-out task's official training data and label the result accordingly. A task withheld from this project's fine-tuning is not proof that its benchmark was absent from the backbone's pretraining. Document that limitation.

Avoid adopting Unsloth's mixed training dataset unchanged for this experiment: the guide includes AG News and Banking77 among its sources. Such a mixture would invalidate a claim that Banking77 was withheld from decision-model adaptation.

## First model: shared candidate scoring

Implement the scalar-scoring design from Chapter 05 using the pretrained decoder. For each question, serialize the state, instruction, candidate label, and description once per candidate. Take each sequence's final non-padding hidden state and apply one shared one-output head. Normalize those scalar scores across that question's candidates.

Train on candidate groups with cross-entropy for the gold option. Keep the number of candidates independent of the head's parameter count. Test changed candidate sets, candidate order, and descriptions. This model is the first transparent implementation and a comparison for the joint head; its repeated backbone work can be costly for many candidates.

A ModernBERT-backed candidate scorer remains an optional encoder comparison using the same data and objective. Its architecture and cost should be reported separately from the decoder experiment.

## Second model: a joint attention head

Use the same decoder checkpoint, but serialize the state and all questions/options into one input. Store token spans during serialization, so every instruction and candidate has a known location. Tokenization must produce the same spans in training and inference. Exclude gold answers from the input and preserve the full schema when truncating a long state.

Follow the derivation in Chapter 06:

1. **Backbone representations.** Read the decoder's final hidden states without computing vocabulary-wide next-token logits. Preserve the backbone's causal attention behavior.
2. **Span pooling.** Pool each instruction and option span and project the full token sequence into the head's input memory. Reject empty spans and exclude padding.
3. **Evidence retrieval.** Build option queries from their descriptions and question instructions. Use `nn.MultiheadAttention` to retrieve evidence from the input memory, with padding masked.
4. **Question interaction.** Form a question vector from its instruction, option summary, and type. Use `nn.TransformerDecoderLayer` for interaction among questions and attention to the input memory. Document the head's attention masks.
5. **Shared scoring.** Apply one shared MLP to each question-option pair. Group the logits by question and normalize only over its allowed options.

This smaller head uses the mechanisms visible in Unsloth's [readable per-record forward pass](https://github.com/unslothai/unsloth/blob/main/unsloth/models/clef.py). The reference includes additional lexical features, similarity priors, scales, and a residual gate. Document every omission, and treat adding those components as a controlled extension. The course model is not an exact Clef reproduction.

Use `nn.Linear`, `nn.LayerNorm`, `nn.MultiheadAttention`, `nn.TransformerDecoderLayer`, ordinary tensor operations, autograd, and an optimizer such as `torch.optim.AdamW`. Existing library acceleration is welcome, but the implementation and its acceptance checks must remain understandable without custom kernels, compiler patches, or fused backward code.

## Training procedure

Begin with a head-only warm-up on a frozen backbone. Overfit a tiny subset and inspect serialization, spans, masks, label mappings, finite losses, and head gradients. This is a pipeline check and a frozen-feature baseline.

Then train the head while adapting the pretrained backbone through LoRA or full fine-tuning. Keep trainable head and backbone/adapter parameters in explicit optimizer groups. Record target modules and adapter rank when using LoRA. Demonstrate that at least one trainable backbone or adapter parameter changes after an optimizer step, as well as a head parameter. A completed capstone must include this adaptation stage.

Start with cross-entropy on each question's allowed options. Define the weighting of records and questions in multi-question batches. As an optional experiment, add a Brier term or an ordinal penalty for Score questions, keeping the comparison splits fixed. The core recipe is supervised fine-tuning; it does not depend on a proprietary calibrated-decision reinforcement-learning algorithm.

Record maximum sequence length, truncation policy, tokenizer and checkpoint revision, head width and layers, trainable parameter counts, optimizer, learning rates, batch size, number of steps or epochs, seeds, and checkpoint-selection rule. Shuffle option order during training with the corresponding target remapping, and audit order sensitivity during evaluation.

## Decision interface

Expose the same operations for the simple scorer and joint head:

- **Choice:** normalize over mutually exclusive candidates; return the selected label and a probability for each allowed label.
- **Noul:** score true and false for each question; return the true probability. Each question has its own distribution, so true probabilities across questions need not sum to one.
- **Score:** return probabilities over an ordered, zero-based rubric and its probability-weighted numeric level. An argmax level is a separate field with a separate meaning.

Question and option counts may vary at inference time. Pad and mask their tensors without changing the shared head's shape. Validate unknown types, empty candidate sets, duplicate labels, empty spans, and schemas that exceed the token budget. Document which state formats are supported and how long inputs are handled.

Treat any separate `confidence` field according to its documented definition. A selected label probability, a concentration summary, and an expected rubric level are different quantities. A Jev-compatible request shape does not establish equivalent confidence semantics or model behavior.

## Calibration and evaluation

Select checkpoints on validation data, fit temperature scaling on the calibration partition, and evaluate test data after both decisions are fixed. Report uncalibrated and calibrated results on the same untouched test rows.

Compare the simple decoder scorer and joint head on the same tasks, data, checkpoint, and training budget. Add the course's TF–IDF/logistic-regression baseline and, where compute allows, a task-specific ModernBERT classifier. Separate testing on trained task families from transfer to the family excluded from adaptation. Label task-specific baselines trained on that target task.

Report accuracy and macro-F1 per task, confusion matrices, and consequential errors. For Score questions, add an ordinal metric such as mean absolute level error. Assess probabilities with log loss, Brier score, and reliability plots or expected calibration error. Report class counts, sample sizes, and mean and variation across at least three seeds when resources allow.

Check candidate-order sensitivity, changed candidate counts, wording changes, and multiple questions per request. A successful flexible interface is not proof of broad task generalization. Identify where training labels, synthetic targets, public benchmark exposure, or distribution shift limit the result.

Measure end-to-end latency for one and several questions/options, examples per second, peak memory, checkpoint size, and hardware. Include tokenization and padding in the measurement and distinguish cold start from steady-state inference.

## Optional optimized benchmark

After establishing the PyTorch model's behavior, compare it with Unsloth as an example of an optimized decision-model system. Its [training guide](https://unsloth.ai/docs/basics/train-your-own-decision-model-with-unsloth), model conversion, trainer, and head implementation provide reference points for the architecture and runtime.

A strict runtime comparison needs the same checkpoint, head, inputs, precision, batching, and hardware. If the educational and reference heads differ, report an end-to-end system comparison rather than attributing the entire latency gap to GPU engineering. Mark provider-reported measurements explicitly. Matching Unsloth's GPU throughput is outside the project requirements.

## Deliverables

1. **Reproducible implementation.** Data adapters, serialization and spans, the simple scorer, joint attention head, explicit training loop, calibration, inference functions, and a CPU smoke example. Keep reusable computation in the project package and use notebooks to interpret it.
2. **Task and data record.** Dataset versions, licenses, provenance, instructions, candidate descriptions, type mappings, split IDs, class counts, record weighting, and the task excluded from adaptation.
3. **Model artifact and card.** Backbone revision, tokenizer, head configuration and weights, adapter weights when used, preprocessing, calibration settings, seeds, selection rule, weights or download instructions, intended uses, and limitations. Verify save/reload predictions.
4. **Evaluation report.** Per-task metrics, calibration, baselines, task-transfer results, errors, repeated-run variation, candidate-order checks, latency, memory, and checkpoint size.
5. **Decision interface.** Example Choice, Noul, and Score requests/responses, input validation, defined probability semantics, and a record of the reference components included or simplified.
6. **Bounded conclusion.** Explain what adaptation changed, whether the joint head helped relative to the simple scorer, what transferred to the held-out task, and which use cases need additional evidence.

## Acceptance review

A reviewer must be able to run the CPU smoke example, trace the span-to-option mapping, inspect the head and loss, and observe a real update to the pretrained backbone or its adapters. Saved artifacts must reproduce the evaluated predictions. Both model variants must return valid distributions for varied candidate sets and multiple Noul questions, and Score must preserve the documented rubric.

The final report must separate trained-task testing from held-out task transfer, compare probabilities as well as labels, and report operational cost. Name the actual pretrained checkpoint and describe the system as an independent Jev-like decision model. Report only experiments actually run; a tiny random-model smoke test does not establish pretrained model quality.

## Extensions

- Add the reference head's lexical prior and gated residual score one component at a time, using the same comparison protocol.
- Rotate the task excluded from adaptation to test sensitivity to the chosen domains.
- Compare cross-entropy-only training with cross-entropy plus a Brier or ordinal penalty, then calibrate on the same reserved partition.
- Serve the trained model behind a local JSON API and audit malformed inputs, long states, and changed schemas.
